In [ ]:
"""
Генератор файлов для лабораторной работы №4.
Тема: Вариация и коэффициент вариации.

Создаёт для каждого из 10 вариантов:
  - lab4_vN_sales.xlsx        — выручка 10 магазинов (Задание 1)
  - lab4_vN_sales_50.xlsx     — 50 магазинов на двух листах (Задания 2, 3)
  - lab4_vN_filials.xlsx      — 3 филиала по 10 сотрудников (Задание 4)

Требуется: pip install openpyxl
"""

import os
import random
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

random.seed(42)  # воспроизводимость

OUT_DIR = "lab4_files"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- Стили ----------
HEADER_FONT = Font(bold=True, color="FFFFFF")
HEADER_FILL = PatternFill("solid", fgColor="4472C4")
CENTER = Alignment(horizontal="center", vertical="center")
THIN = Side(style="thin", color="999999")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)


def style_header(ws, ncols):
    for col in range(1, ncols + 1):
        c = ws.cell(row=1, column=col)
        c.font = HEADER_FONT
        c.fill = HEADER_FILL
        c.alignment = CENTER
        c.border = BORDER


def autofit(ws, widths):
    for i, w in enumerate(widths, start=1):
        ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = w


# ============================================================
# 1. ДАННЫЕ ДЛЯ ВАРИАНТОВ (файл 1 — 10 магазинов)
# ============================================================
# Каждый вариант: (город, [9 нормальных значений], выброс_значение, индекс_выброса)
# Индексы: 1..10, выброс обычно на позиции 10 или 9.

VARIANTS_FILE1 = {
    1:  ("Москва",           [1_000_000, 1_050_000, 980_000, 1_020_000, 1_010_000,
                              990_000, 1_030_000, 1_040_000, 1_000_000], 2_500_000, 10),
    2:  ("СПб",              [2_000_000, 2_100_000, 1_950_000, 2_050_000, 2_000_000,
                              2_100_000, 1_980_000, 2_020_000, 2_000_000], 200_000, 9),
    3:  ("Екатеринбург",     [5_000_000, 5_200_000, 4_800_000, 5_100_000, 5_000_000,
                              5_300_000, 4_900_000, 5_050_000, 5_150_000], 15_000_000, 10),
    4:  ("Новосибирск",      [300_000, 310_000, 290_000, 305_000, 295_000,
                              300_000, 310_000, 290_000, 300_000], 3_000_000, 10),
    5:  ("Казань",           [800_000, 820_000, 780_000, 810_000, 790_000,
                              800_000, 820_000, 780_000, 800_000], 80_000, 10),
    6:  ("Москва",           [10_000_000, 10_500_000, 9_500_000, 10_200_000, 9_800_000,
                              10_000_000, 10_300_000, 9_700_000, 10_000_000], 50_000_000, 10),
    7:  ("Нижний Новгород",  [1_500_000, 1_550_000, 1_450_000, 1_520_000, 1_480_000,
                              1_500_000, 1_530_000, 1_470_000, 1_500_000], 8_000_000, 10),
    8:  ("Краснодар",        [400_000, 410_000, 390_000, 405_000, 395_000,
                              400_000, 410_000, 390_000, 400_000], 100_000, 10),
    9:  ("Самара",           [2_500_000, 2_550_000, 2_450_000, 2_520_000, 2_480_000,
                              2_500_000, 2_530_000, 2_470_000, 2_500_000], 12_000_000, 10),
    10: ("Москва",           [600_000, 620_000, 580_000, 610_000, 590_000,
                              600_000, 620_000, 580_000, 600_000], 6_000_000, 10),
}


# ============================================================
# 2. ДАННЫЕ ДЛЯ ВАРИАНТОВ (файл 2 — 50 магазинов, 2 листа)
# ============================================================
# (centr_stable, half_stable, centr_unstable, half_unstable)
# half — половина разброса вокруг центра.
# Для стабильной: V ≈ (half/centr)*100% / sqrt(3) примерно ~3-5%
# Для нестабильной: V ≈ 50-70% — задаём широкий разброс.

VARIANTS_FILE2 = {
    1:  (1_000_000, 30_000,  1_850_000, 1_000_000),  # V_st ~3%, V_un ~65%
    2:  (2_000_000, 60_000,  2_250_000, 1_200_000),  # V_st ~3%, V_un ~55%
    3:  (5_000_000, 150_000, 6_500_000, 4_500_000),  # V_st ~3%, V_un ~70%
    4:  (300_000,   15_000,  450_000,   250_000),    # V_st ~5%, V_un ~60%
    5:  (800_000,   15_000,  1_100_000, 700_000),    # V_st ~2%, V_un ~65%
    6:  (10_000_000, 300_000, 17_500_000, 8_000_000),# V_st ~3%, V_un ~50%
    7:  (1_500_000, 45_000,  2_750_000, 1_500_000),  # V_st ~3%, V_un ~60%
    8:  (400_000,   20_000,  550_000,   350_000),    # V_st ~5%, V_un ~65%
    9:  (2_500_000, 100_000, 4_500_000, 3_000_000),  # V_st ~4%, V_un ~70%
    10: (600_000,   18_000,  1_100_000, 600_000),    # V_st ~3%, V_un ~55%
}


def gen_stable(center, half, n=50):
    """50 значений равномерно в [center-half, center+half]."""
    return [random.randint(center - half, center + half) for _ in range(n)]


def gen_unstable(center, half, n=50):
    """50 значений с широким разбросом; смесь двух диапазонов для реалистичности."""
    low = center - half
    high = center + half
    # разбиваем на 2 группы, чтобы получить ненулевой эксцесс и правдоподобность
    result = []
    for _ in range(n):
        if random.random() < 0.5:
            result.append(random.randint(low, center))
        else:
            result.append(random.randint(center, high))
    random.shuffle(result)
    return result


# ============================================================
# 3. ДАННЫЕ ДЛЯ ВАРИАНТОВ (файл 3 — 3 филиала × 10 сотрудников)
# ============================================================
# (mean1, sigma1, mean2, sigma2, mean3, sigma3)
# Согласно описанию:
# 1: средние разные (500, 700, 300), sigma ~7 000
# 2: средние одинаковые (400), sigma = 5k, 100k, 5k
# 3: средние разные (800, 1200, 600), sigma маленький
# 4: средние одинаковые (200), sigma = 50k у всех
# 5: средние разные (500, 700, 500), sigma: мал, большой, мал
# 6: средние одинаковые (2 000 000), sigma = 500k у всех
# 7: средние разные (600, 900, 300), sigma маленький
# 8: средние одинаковые (300k), sigma большой у всех
# 9: средние разные (1 000 000, 1 500 000, 500 000), sigma: мал, большой, мал
# 10: средние одинаковые (400k), sigma маленький

VARIANTS_FILE3 = {
    1:  (500_000, 7_000, 700_000, 7_000, 300_000, 7_000),
    2:  (400_000, 5_000, 400_000, 100_000, 400_000, 5_000),
    3:  (800_000, 10_000, 1_200_000, 10_000, 600_000, 10_000),
    4:  (200_000, 50_000, 200_000, 50_000, 200_000, 50_000),
    5:  (500_000, 7_000, 700_000, 80_000, 500_000, 7_000),
    6:  (2_000_000, 500_000, 2_000_000, 500_000, 2_000_000, 500_000),
    7:  (600_000, 15_000, 900_000, 15_000, 300_000, 15_000),
    8:  (300_000, 60_000, 300_000, 60_000, 300_000, 60_000),
    9:  (1_000_000, 20_000, 1_500_000, 150_000, 500_000, 20_000),
    10: (400_000, 8_000, 400_000, 8_000, 400_000, 8_000),
}


def gen_filial(mean, sigma, n=10):
    """Генерирует n значений ~ N(mean, sigma) с округлением до тысяч."""
    vals = [max(0, int(random.gauss(mean, sigma))) for _ in range(n)]
    return [round(v, -3) for v in vals]  # округление до тысяч


# ============================================================
# СОЗДАНИЕ ФАЙЛА 1
# ============================================================
def create_file1(vnum):
    city, normals, outlier, out_idx = VARIANTS_FILE1[vnum]
    revenues = []
    ni = 0
    for i in range(1, 11):
        if i == out_idx:
            revenues.append(outlier)
        else:
            revenues.append(normals[ni])
            ni += 1

    wb = Workbook()
    ws = wb.active
    ws.title = "Выручка"
    ws.append(["№", "Город", "Выручка, руб."])
    for i, rev in enumerate(revenues, start=1):
        ws.append([i, city, rev])
    style_header(ws, 3)
    autofit(ws, [8, 20, 18])
    for row in ws.iter_rows(min_row=2, min_col=3, max_col=3):
        for c in row:
            c.number_format = "#,##0"

    path = os.path.join(OUT_DIR, f"lab4_v{vnum}_sales.xlsx")
    wb.save(path)
    print(f"  ✓ {path}")


# ============================================================
# СОЗДАНИЕ ФАЙЛА 2 (2 листа)
# ============================================================
def create_file2(vnum):
    c_st, h_st, c_un, h_un = VARIANTS_FILE2[vnum]
    stable = gen_stable(c_st, h_st, 50)
    unstable = gen_unstable(c_un, h_un, 50)

    wb = Workbook()
    ws1 = wb.active
    ws1.title = "Стабильная сеть"
    ws1.append(["№", "Выручка, руб."])
    for i, v in enumerate(stable, start=1):
        ws1.append([i, v])
    style_header(ws1, 2)
    autofit(ws1, [8, 18])
    for row in ws1.iter_rows(min_row=2, min_col=2, max_col=2):
        for c in row:
            c.number_format = "#,##0"

    ws2 = wb.create_sheet("Нестабильная сеть")
    ws2.append(["№", "Выручка, руб."])
    for i, v in enumerate(unstable, start=1):
        ws2.append([i, v])
    style_header(ws2, 2)
    autofit(ws2, [8, 18])
    for row in ws2.iter_rows(min_row=2, min_col=2, max_col=2):
        for c in row:
            c.number_format = "#,##0"

    path = os.path.join(OUT_DIR, f"lab4_v{vnum}_sales_50.xlsx")
    wb.save(path)
    print(f"  ✓ {path}")


# ============================================================
# СОЗДАНИЕ ФАЙЛА 3 (3 филиала × 10 сотрудников)
# ============================================================
def create_file3(vnum):
    m1, s1, m2, s2, m3, s3 = VARIANTS_FILE3[vnum]
    f1 = gen_filial(m1, s1)
    f2 = gen_filial(m2, s2)
    f3 = gen_filial(m3, s3)

    wb = Workbook()
    ws = wb.active
    ws.title = "Филиалы"
    ws.append(["Филиал", "Сотрудник", "Выручка, руб."])

    for filial_name, data in [("Филиал 1", f1), ("Филиал 2", f2), ("Филиал 3", f3)]:
        for i, v in enumerate(data, start=1):
            ws.append([filial_name, i, v])

    style_header(ws, 3)
    autofit(ws, [14, 12, 18])
    for row in ws.iter_rows(min_row=2, min_col=3, max_col=3):
        for c in row:
            c.number_format = "#,##0"

    path = os.path.join(OUT_DIR, f"lab4_v{vnum}_filials.xlsx")
    wb.save(path)
    print(f"  ✓ {path}")


# ============================================================
# ГЛАВНЫЙ ЦИКЛ
# ============================================================
def main():
    print(f"Создание файлов в папке: {os.path.abspath(OUT_DIR)}\n")
    for v in range(1, 11):
        print(f"Вариант {v}:")
        create_file1(v)
        create_file2(v)
        create_file3(v)
        print()
    print("Готово! Создано 30 файлов.")


if __name__ == "__main__":
    main()

In [ ]:
pip install openpyxl

In [ ]:
"""
Генератор одной книги Excel со всеми вариантами лабораторной работы №4.
Тема: Вариация и коэффициент вариации.

Структура книги lab4_all_variants.xlsx:
  - Задание1_Вариант1..10  — 10 магазинов (Задание 1)
  - Задание2_Вариант1..10  — 50 магазинов: Сеть 1 и Сеть 2 (Задания 2, 3)
  - Задание3_Вариант1..10  — 3 филиала × 10 сотрудников (Задание 4)

Требуется: pip install openpyxl
"""

import os
import random
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

random.seed(42)  # воспроизводимость

OUT_FILE = "lab4_all_variants.xlsx"

# ---------- Стили ----------
HEADER_FONT = Font(bold=True, color="FFFFFF")
HEADER_FILL = PatternFill("solid", fgColor="4472C4")
HEADER_FILL2 = PatternFill("solid", fgColor="C00000")  # для 2-й сети
HEADER_FILL3 = PatternFill("solid", fgColor="548235")  # для филиалов
CENTER = Alignment(horizontal="center", vertical="center")
THIN = Side(style="thin", color="999999")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)


def style_header(ws, row, col_start, col_end, fill=HEADER_FILL):
    for col in range(col_start, col_end + 1):
        c = ws.cell(row=row, column=col)
        c.font = HEADER_FONT
        c.fill = fill
        c.alignment = CENTER
        c.border = BORDER


def set_widths(ws, widths, start_col=1):
    for i, w in enumerate(widths, start=start_col):
        ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = w


def fmt_numbers(ws, row_start, row_end, col_start, col_end):
    for r in range(row_start, row_end + 1):
        for c in range(col_start, col_end + 1):
            ws.cell(row=r, column=c).number_format = "#,##0"


# ============================================================
# 1. ДАННЫЕ ДЛЯ ВАРИАНТОВ (Задание 1 — 10 магазинов)
# ============================================================
VARIANTS_FILE1 = {
    1:  ("Москва",           [1_000_000, 1_050_000, 980_000, 1_020_000, 1_010_000,
                              990_000, 1_030_000, 1_040_000, 1_000_000], 2_500_000, 10),
    2:  ("СПб",              [2_000_000, 2_100_000, 1_950_000, 2_050_000, 2_000_000,
                              2_100_000, 1_980_000, 2_020_000, 2_000_000], 200_000, 9),
    3:  ("Екатеринбург",     [5_000_000, 5_200_000, 4_800_000, 5_100_000, 5_000_000,
                              5_300_000, 4_900_000, 5_050_000, 5_150_000], 15_000_000, 10),
    4:  ("Новосибирск",      [300_000, 310_000, 290_000, 305_000, 295_000,
                              300_000, 310_000, 290_000, 300_000], 3_000_000, 10),
    5:  ("Казань",           [800_000, 820_000, 780_000, 810_000, 790_000,
                              800_000, 820_000, 780_000, 800_000], 80_000, 10),
    6:  ("Москва",           [10_000_000, 10_500_000, 9_500_000, 10_200_000, 9_800_000,
                              10_000_000, 10_300_000, 9_700_000, 10_000_000], 50_000_000, 10),
    7:  ("Нижний Новгород",  [1_500_000, 1_550_000, 1_450_000, 1_520_000, 1_480_000,
                              1_500_000, 1_530_000, 1_470_000, 1_500_000], 8_000_000, 10),
    8:  ("Краснодар",        [400_000, 410_000, 390_000, 405_000, 395_000,
                              400_000, 410_000, 390_000, 400_000], 100_000, 10),
    9:  ("Самара",           [2_500_000, 2_550_000, 2_450_000, 2_520_000, 2_480_000,
                              2_500_000, 2_530_000, 2_470_000, 2_500_000], 12_000_000, 10),
    10: ("Москва",           [600_000, 620_000, 580_000, 610_000, 590_000,
                              600_000, 620_000, 580_000, 600_000], 6_000_000, 10),
}


# ============================================================
# 2. ДАННЫЕ ДЛЯ ВАРИАНТОВ (Задание 2 — 50 магазинов, две сети)
# ============================================================
VARIANTS_FILE2 = {
    1:  (1_000_000, 30_000,  1_850_000, 1_000_000),
    2:  (2_000_000, 60_000,  2_250_000, 1_200_000),
    3:  (5_000_000, 150_000, 6_500_000, 4_500_000),
    4:  (300_000,   15_000,  450_000,   250_000),
    5:  (800_000,   15_000,  1_100_000, 700_000),
    6:  (10_000_000, 300_000, 17_500_000, 8_000_000),
    7:  (1_500_000, 45_000,  2_750_000, 1_500_000),
    8:  (400_000,   20_000,  550_000,   350_000),
    9:  (2_500_000, 100_000, 4_500_000, 3_000_000),
    10: (600_000,   18_000,  1_100_000, 600_000),
}


def gen_network1(center, half, n=50):
    return [random.randint(center - half, center + half) for _ in range(n)]


def gen_network2(center, half, n=50):
    low, high = center - half, center + half
    result = []
    for _ in range(n):
        if random.random() < 0.5:
            result.append(random.randint(low, center))
        else:
            result.append(random.randint(center, high))
    random.shuffle(result)
    return result


# ============================================================
# 3. ДАННЫЕ ДЛЯ ВАРИАНТОВ (Задание 3 — 3 филиала × 10 сотрудников)
# ============================================================
VARIANTS_FILE3 = {
    1:  (500_000, 7_000, 700_000, 7_000, 300_000, 7_000),
    2:  (400_000, 5_000, 400_000, 100_000, 400_000, 5_000),
    3:  (800_000, 10_000, 1_200_000, 10_000, 600_000, 10_000),
    4:  (200_000, 50_000, 200_000, 50_000, 200_000, 50_000),
    5:  (500_000, 7_000, 700_000, 80_000, 500_000, 7_000),
    6:  (2_000_000, 500_000, 2_000_000, 500_000, 2_000_000, 500_000),
    7:  (600_000, 15_000, 900_000, 15_000, 300_000, 15_000),
    8:  (300_000, 60_000, 300_000, 60_000, 300_000, 60_000),
    9:  (1_000_000, 20_000, 1_500_000, 150_000, 500_000, 20_000),
    10: (400_000, 8_000, 400_000, 8_000, 400_000, 8_000),
}


def gen_filial(mean, sigma, n=10):
    vals = [max(0, int(random.gauss(mean, sigma))) for _ in range(n)]
    return [round(v, -3) for v in vals]


# ============================================================
# СОЗДАНИЕ ЛИСТОВ
# ============================================================
def create_task1_sheet(wb, vnum):
    city, normals, outlier, out_idx = VARIANTS_FILE1[vnum]
    revenues = []
    ni = 0
    for i in range(1, 11):
        if i == out_idx:
            revenues.append(outlier)
        else:
            revenues.append(normals[ni])
            ni += 1

    ws = wb.create_sheet(f"Задание1_Вариант{vnum}")
    ws.append(["№", "Город", "Выручка, руб."])
    for i, rev in enumerate(revenues, start=1):
        ws.append([i, city, rev])

    style_header(ws, 1, 1, 3)
    set_widths(ws, [8, 20, 18])
    fmt_numbers(ws, 2, 11, 3, 3)


def create_task2_sheet(wb, vnum):
    c1, h1, c2, h2 = VARIANTS_FILE2[vnum]
    net1 = gen_network1(c1, h1, 50)
    net2 = gen_network2(c2, h2, 50)

    ws = wb.create_sheet(f"Задание2_Вариант{vnum}")

    # Заголовки блоков
    ws.cell(row=1, column=1, value="Сеть 1").font = Font(bold=True, size=12)
    ws.cell(row=1, column=4, value="Сеть 2").font = Font(bold=True, size=12)
    ws.cell(row=1, column=1).alignment = CENTER
    ws.cell(row=1, column=4).alignment = CENTER

    # Шапка таблиц
    ws.cell(row=2, column=1, value="№")
    ws.cell(row=2, column=2, value="Сеть 1, руб.")
    ws.cell(row=2, column=4, value="№")
    ws.cell(row=2, column=5, value="Сеть 2, руб.")

    style_header(ws, 2, 1, 2, HEADER_FILL)
    style_header(ws, 2, 4, 5, HEADER_FILL2)

    for i in range(50):
        ws.cell(row=3 + i, column=1, value=i + 1)
        ws.cell(row=3 + i, column=2, value=net1[i])
        ws.cell(row=3 + i, column=4, value=i + 1)
        ws.cell(row=3 + i, column=5, value=net2[i])

    set_widths(ws, [8, 16, 3, 8, 16])
    fmt_numbers(ws, 3, 52, 2, 2)
    fmt_numbers(ws, 3, 52, 5, 5)


def create_task3_sheet(wb, vnum):
    m1, s1, m2, s2, m3, s3 = VARIANTS_FILE3[vnum]
    f1 = gen_filial(m1, s1)
    f2 = gen_filial(m2, s2)
    f3 = gen_filial(m3, s3)

    ws = wb.create_sheet(f"Задание3_Вариант{vnum}")
    ws.append(["Филиал", "Сотрудник", "Выручка, руб."])

    for filial_name, data in [("Филиал 1", f1), ("Филиал 2", f2), ("Филиал 3", f3)]:
        for i, v in enumerate(data, start=1):
            ws.append([filial_name, i, v])

    style_header(ws, 1, 1, 3, HEADER_FILL3)
    set_widths(ws, [14, 12, 18])
    fmt_numbers(ws, 2, 31, 3, 3)


# ============================================================
# ГЛАВНЫЙ ЦИКЛ
# ============================================================
def main():
    wb = Workbook()
    wb.remove(wb.active)  # удаляем пустой лист по умолчанию

    for v in range(1, 11):
        create_task1_sheet(wb, v)

    for v in range(1, 11):
        create_task2_sheet(wb, v)

    for v in range(1, 11):
        create_task3_sheet(wb, v)

    wb.save(OUT_FILE)
    print(f"Готово! Создан файл: {os.path.abspath(OUT_FILE)}")
    print(f"Всего листов: {len(wb.sheetnames)}")
    for name in wb.sheetnames:
        print(f"  • {name}")


if __name__ == "__main__":
    main()

In [ ]:
pip install openpyxl python-docx

In [ ]:
"""
Читает lab4_all_variants.xlsx и формирует проверочную записку
lab4_reference_answers.docx с расчётами по всем вариантам.

Задания:
  1. Размах, дисперсия, σ, средняя, поиск выброса (по σ от средней).
  2. Коэффициент вариации для Сети 1 и Сети 2 + оценка однородности.
  3. Правило трёх сигм — поиск аномалий в Сети 1.
  4. Правило сложения дисперсий для 3 филиалов.
  5. Итоговый вывод для службы безопасности.
"""

import os
import statistics as st
from openpyxl import load_workbook
from docx import Document
from docx.shared import Pt, RGBColor, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_ALIGN_VERTICAL

SRC = "lab4_all_variants.xlsx"
OUT = "lab4_reference_answers.docx"
VARIANTS = range(1, 11)


# ============================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================
def fmt(x):
    """Красивое число с разделителями тысяч."""
    if isinstance(x, float):
        return f"{x:,.2f}".replace(",", " ")
    return f"{x:,}".replace(",", " ")


def pct(x):
    return f"{x:.2f}%"


def add_heading(doc, text, level=1):
    h = doc.add_heading(text, level=level)
    return h


def add_table(doc, headers, rows, style="Light Grid Accent 1"):
    t = doc.add_table(rows=1, cols=len(headers))
    t.style = style
    hdr = t.rows[0].cells
    for i, h in enumerate(headers):
        hdr[i].text = str(h)
        for p in hdr[i].paragraphs:
            for r in p.runs:
                r.bold = True
    for row in rows:
        cells = t.add_row().cells
        for i, val in enumerate(row):
            cells[i].text = str(val)
    return t


# ============================================================
# ЧТЕНИЕ ДАННЫХ ИЗ EXCEL
# ============================================================
def read_task1(wb, vnum):
    ws = wb[f"Задание1_Вариант{vnum}"]
    rows = []
    for r in range(2, 12):
        rows.append({
            "num": ws.cell(r, 1).value,
            "city": ws.cell(r, 2).value,
            "rev": ws.cell(r, 3).value,
        })
    return rows


def read_task2(wb, vnum):
    ws = wb[f"Задание2_Вариант{vnum}"]
    net1, net2 = [], []
    for r in range(3, 53):
        n1 = ws.cell(r, 1).value
        v1 = ws.cell(r, 2).value
        if n1 is not None and v1 is not None:
            net1.append(v1)
        n2 = ws.cell(r, 4).value
        v2 = ws.cell(r, 5).value
        if n2 is not None and v2 is not None:
            net2.append(v2)
    return net1, net2


def read_task3(wb, vnum):
    ws = wb[f"Задание3_Вариант{vnum}"]
    filials = {"Филиал 1": [], "Филиал 2": [], "Филиал 3": []}
    for r in range(2, 32):
        f = ws.cell(r, 1).value
        v = ws.cell(r, 3).value
        if f in filials and v is not None:
            filials[f].append(v)
    return filials


# ============================================================
# РАСЧЁТЫ
# ============================================================
def calc_task1(rows):
    revs = [r["rev"] for r in rows]
    res = {
        "min": min(revs),
        "max": max(revs),
        "range": max(revs) - min(revs),
        "mean": st.mean(revs),
        "var_g": st.pvariance(revs),      # генеральная дисперсия (ДИСП.Г)
        "var_s": st.variance(revs),       # выборочная (ДИСП.В)
        "sigma_g": st.pstdev(revs),
        "sigma_s": st.stdev(revs),
    }
    res["V_g"] = res["sigma_g"] / res["mean"] * 100
    res["V_s"] = res["sigma_s"] / res["mean"] * 100

    # поиск наиболее отклоняющегося магазина
    devs = [abs(r["rev"] - res["mean"]) for r in rows]
    idx = devs.index(max(devs))
    res["outlier"] = rows[idx]
    res["outlier_dev"] = rows[idx]["rev"] - res["mean"]
    res["outlier_dev_pct"] = res["outlier_dev"] / res["mean"] * 100
    # какой магазин «выбивается» (в 3σ?)
    res["beyond_3s"] = [r for r in rows
                        if abs(r["rev"] - res["mean"]) > 3 * res["sigma_g"]]
    return res


def calc_network(data):
    m = st.mean(data)
    s_g = st.pstdev(data)
    s_s = st.stdev(data)
    return {
        "n": len(data),
        "mean": m,
        "sigma_g": s_g,
        "sigma_s": s_s,
        "V_g": s_g / m * 100,
        "V_s": s_s / m * 100,
        "min": min(data),
        "max": max(data),
    }


def assess_V(V):
    if V < 33:
        return "однородная ✅"
    elif V <= 67:
        return "средняя неоднородность ⚠️"
    else:
        return "неоднородная 🔴"


def find_anomalies_3sigma(data):
    m = st.mean(data)
    s = st.pstdev(data)
    low, high = m - 3 * s, m + 3 * s
    anomalies = []
    for i, v in enumerate(data, start=1):
        if v < low or v > high:
            anomalies.append({
                "num": i,
                "value": v,
                "dev": v - m,
                "dev_pct": (v - m) / m * 100,
                "side": "ниже нижней" if v < low else "выше верхней",
            })
    return low, high, anomalies


def calc_task4(filials):
    groups = list(filials.values())
    means = [st.mean(g) for g in groups]
    variances = [st.pvariance(g) for g in groups]
    ns = [len(g) for g in groups]
    total_n = sum(ns)

    # средняя внутригрупповая
    var_within = sum(n * v for n, v in zip(ns, variances)) / total_n
    # общая средняя
    grand_mean = sum(n * m for n, m in zip(ns, means)) / total_n
    # межгрупповая
    var_between = sum(n * (m - grand_mean) ** 2 for n, m in zip(ns, means)) / total_n
    # общая
    all_vals = [v for g in groups for v in g]
    var_total = st.pvariance(all_vals)

    return {
        "means": means,
        "variances": variances,
        "var_within": var_within,
        "var_between": var_between,
        "var_total": var_total,
        "grand_mean": grand_mean,
        "check": abs(var_within + var_between - var_total) < 1e-3,
        "share_within": var_within / var_total * 100,
        "share_between": var_between / var_total * 100,
    }


# ============================================================
# ФОРМИРОВАНИЕ ДОКУМЕНТА
# ============================================================
def write_task1(doc, vnum, r):
    add_heading(doc, f"ЗАДАНИЕ 1. Размах и дисперсия (Вариант {vnum})", level=2)

    doc.add_paragraph(
        f"Исходные данные: 10 магазинов. Диапазон выручки: "
        f"{fmt(r['min'])} – {fmt(r['max'])} руб."
    )

    add_table(doc,
              ["Показатель", "Значение"],
              [
                  ["Минимум", fmt(r["min"]) + " руб."],
                  ["Максимум", fmt(r["max"]) + " руб."],
                  ["Размах вариации", fmt(r["range"]) + " руб."],
                  ["Средняя", fmt(r["mean"]) + " руб."],
                  ["Дисперсия (ген.)", fmt(r["var_g"])],
                  ["Дисперсия (выб.)", fmt(r["var_s"])],
                  ["σ (ген.)", fmt(r["sigma_g"]) + " руб."],
                  ["σ (выб.)", fmt(r["sigma_s"]) + " руб."],
                  ["Коэффициент вариации V (ген.)", pct(r["V_g"])],
              ])

    doc.add_paragraph()

    # Вывод по выбросу
    o = r["outlier"]
    doc.add_paragraph(
        f"Наиболее отклоняющийся магазин — №{o['num']} ({o['city']}), "
        f"выручка {fmt(o['rev'])} руб. "
        f"Отклонение от средней: {fmt(o['outlier_dev'])} руб. "
        f"({pct(r['outlier_dev_pct'])})."
    )

    if r["beyond_3s"]:
        for x in r["beyond_3s"]:
            doc.add_paragraph(
                f"❗ Магазин №{x['num']} выходит за пределы 3σ "
                f"(выручка {fmt(x['rev'])} руб.). Это аномалия."
            )
    else:
        doc.add_paragraph(
            "Магазинов, выходящих за пределы 3σ, не обнаружено, "
            "но есть явный выброс, искажающий среднюю."
        )

    doc.add_paragraph(
        "Вывод: разброс выручки очень велик (σ составляет "
        f"{pct(r['sigma_g'] / r['mean'] * 100)} от средней). "
        "Это означает, что один магазин радикально отличается от остальных. "
        f"Требуется проверка магазина №{o['num']}."
    )


def write_task2(doc, vnum, n1, n2):
    add_heading(doc, f"ЗАДАНИЕ 2. Коэффициент вариации (Вариант {vnum})", level=2)

    add_table(doc,
              ["Группа", "n", "Средняя", "σ (ген.)", "V, %", "Оценка"],
              [
                  ["Сеть 1", n1["n"], fmt(n1["mean"]), fmt(n1["sigma_g"]),
                   pct(n1["V_g"]), assess_V(n1["V_g"])],
                  ["Сеть 2", n2["n"], fmt(n2["mean"]), fmt(n2["sigma_g"]),
                   pct(n2["V_g"]), assess_V(n2["V_g"])],
              ])

    doc.add_paragraph()
    doc.add_paragraph(
        f"Вывод: коэффициент вариации в Сети 1 составляет {pct(n1['V_g'])} — "
        f"совокупность {assess_V(n1['V_g'])}. "
        f"В Сети 2 V = {pct(n2['V_g'])} — совокупность {assess_V(n2['V_g'])}. "
        f"Сеть 2 требует первоочередной проверки: высокий V означает, "
        "что внутри сети есть магазины с принципиально разным поведением, "
        "что может быть признаком манипуляций с выручкой."
    )


def write_task3(doc, vnum, net1):
    add_heading(doc, f"ЗАДАНИЕ 3. Правило трёх сигм (Вариант {vnum}, Сеть 1)", level=2)

    low, high, anomalies = find_anomalies_3sigma(net1)
    m = st.mean(net1)
    s = st.pstdev(net1)

    add_table(doc,
              ["Показатель", "Значение"],
              [
                  ["Средняя", fmt(m) + " руб."],
                  ["σ", fmt(s) + " руб."],
                  ["Нижняя граница (μ − 3σ)", fmt(low) + " руб."],
                  ["Верхняя граница (μ + 3σ)", fmt(high) + " руб."],
                  ["Количество аномалий", len(anomalies)],
              ])

    doc.add_paragraph()

    if anomalies:
        rows = [[a["num"], fmt(a["value"]),
                 fmt(a["dev"]), pct(a["dev_pct"]), a["side"]]
                for a in anomalies]
        add_table(doc,
                  ["№ магазина", "Выручка", "Отклонение от средней",
                   "Отклонение, %", "Сторона"],
                  rows)
        doc.add_paragraph()
        for a in anomalies:
            doc.add_paragraph(
                f"Магазин №{a['num']}: выручка {fmt(a['value'])} руб., "
                f"что на {pct(a['dev_pct'])} {'выше' if a['dev'] > 0 else 'ниже'} средней. "
                "Требует проверки."
            )
    else:
        doc.add_paragraph(
            "Аномальных магазинов по правилу трёх сигм не выявлено — "
            "Сеть 1 работает стабильно."
        )

    doc.add_paragraph(
        "Рекомендация службе безопасности: "
        + ("проверить выявленные аномальные магазины на предмет "
           "манипуляций с выручкой, ошибок учёта или хищений."
           if anomalies else
           "продолжать мониторинг, отклонений не обнаружено.")
    )


def write_task4(doc, vnum, filials):
    add_heading(doc, f"ЗАДАНИЕ 4. Правило сложения дисперсий (Вариант {vnum})", level=2)

    r = calc_task4(filials)

    add_table(doc,
              ["Филиал", "Средняя выручка", "Внутригрупповая дисперсия"],
              [[f"Филиал {i+1}", fmt(r["means"][i]), fmt(r["variances"][i])]
               for i in range(3)])

    doc.add_paragraph()
    add_table(doc,
              ["Компонент дисперсии", "Значение", "Доля, %"],
              [
                  ["Внутригрупповая", fmt(r["var_within"]), pct(r["share_within"])],
                  ["Межгрупповая", fmt(r["var_between"]), pct(r["share_between"])],
                  ["Общая", fmt(r["var_total"]), "100.00%"],
              ])

    doc.add_paragraph()
    doc.add_paragraph(
        f"Проверка правила сложения: "
        f"{fmt(r['var_within'])} + {fmt(r['var_between'])} = "
        f"{fmt(r['var_within'] + r['var_between'])} ≈ {fmt(r['var_total'])} "
        f"({'✅ выполняется' if r['check'] else '❌ НЕ выполняется'})."
    )

    doc.add_paragraph()
    if r["var_between"] > r["var_within"]:
        doc.add_paragraph(
            "Вывод: межгрупповая дисперсия больше внутригрупповой. "
            "Проблема системная — филиалы работают по-разному. "
            "Рекомендуется пересмотреть политику управления филиалами."
        )
    else:
        doc.add_paragraph(
            "Вывод: внутригрупповая дисперсия больше межгрупповой. "
            "Проблема локальная — внутри филиалов сотрудники работают "
            "по-разному. Рекомендуется разбираться с конкретными сотрудниками."
        )


def write_task5(doc, vnum, t1, n1, n2, net1, filials):
    add_heading(doc, f"ЗАДАНИЕ 5. Вывод для службы безопасности (Вариант {vnum})", level=2)

    low, high, anomalies = find_anomalies_3sigma(net1)
    r4 = calc_task4(filials)

    if r4["var_between"] > r4["var_within"]:
        source = "межгрупповая"
        source_text = ("проблема на уровне филиалов — нужно менять "
                       "политику управления")
    else:
        source = "внутригрупповая"
        source_text = ("проблема на уровне отдельных сотрудников — "
                       "нужно разбираться с конкретными людьми")

    anomaly_text = ", ".join([f"№{a['num']} ({fmt(a['value'])} руб.)"
                              for a in anomalies]) or "нет"

    text = (
        f"По результатам анализа вариации выявлено: коэффициент вариации "
        f"по Сети 1 составил {pct(n1['V_g'])}, по Сети 2 — {pct(n2['V_g'])}. "
        f"Это означает, что Сеть 2 {assess_V(n2['V_g'])}. "
        f"По правилу трёх сигм в Сети 1 выявлено {len(anomalies)} "
        f"аномальных магазинов: {anomaly_text}. "
        f"В Задании 1 наиболее отклоняющийся магазин — "
        f"№{t1['outlier']['num']} ({fmt(t1['outlier']['rev'])} руб.). "
        f"Правило сложения дисперсий показало, что основной источник "
        f"разброса — {source} дисперсия ({pct(r4['share_within'] if source == 'внутригрупповая' else r4['share_between'])}), "
        f"что говорит о том, что {source_text}. "
        f"Рекомендуется проверить выявленные аномальные магазины и "
        f"нестабильные филиалы на предмет манипуляций с выручкой."
    )
    doc.add_paragraph(text)


# ============================================================
# ГЛАВНАЯ ФУНКЦИЯ
# ============================================================
def main():
    if not os.path.exists(SRC):
        raise FileNotFoundError(
            f"Не найден файл {SRC}. Сначала запустите скрипт генерации."
        )

    wb = load_workbook(SRC, data_only=True)
    doc = Document()

    # Титульный лист
    title = doc.add_heading("Лабораторная работа №4", level=0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    sub = doc.add_paragraph("Вариация и коэффициент вариации")
    sub.alignment = WD_ALIGN_PARAGRAPH.CENTER
    sub.runs[0].bold = True
    sub.runs[0].font.size = Pt(16)

    doc.add_paragraph()
    doc.add_paragraph(
        "Проверочная записка с эталонными расчётами по всем 10 вариантам. "
        "Документ предназначен для проверки студенческих работ."
    )
    doc.add_page_break()

    for v in VARIANTS:
        add_heading(doc, f"ВАРИАНТ {v}", level=1)

        # Задание 1
        rows1 = read_task1(wb, v)
        t1 = calc_task1(rows1)
        write_task1(doc, v, t1)

        # Задание 2
        net1_data, net2_data = read_task2(wb, v)
        n1 = calc_network(net1_data)
        n2 = calc_network(net2_data)
        write_task2(doc, v, n1, n2)

        # Задание 3
        write_task3(doc, v, net1_data)

        # Задание 4
        filials = read_task3(wb, v)
        write_task4(doc, v, filials)

        # Задание 5
        write_task5(doc, v, t1, n1, n2, net1_data, filials)

        doc.add_page_break()

    doc.save(OUT)
    print(f"Готово! Создан файл: {os.path.abspath(OUT)}")
    print(f"Вариантов обработано: {len(list(VARIANTS))}")


if __name__ == "__main__":
    main()